# Pleeb — Colab Large-Model Benchmark

Tests Whisper `small` / `medium` / `large` with the full Pleeb timestamp pipeline.

**Runtime:** GPU (T4 or better). CPU will work but large takes ~10×.

**What this does:**
- Installs all dependencies with pinned versions
- Accepts an uploaded audio or video file
- Runs each model sequentially, evicting the previous from VRAM before loading the next
- Displays runtime, word count, and average confidence per model
- Exports a JSON benchmark artifact

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Pinned versions prevent silent breakage when upstream releases change APIs.

!apt-get -qq update
!apt-get -qq install -y ffmpeg

!pip -q install --upgrade pip

# PyTorch: CUDA 12.1 wheel works on Colab T4 (CUDA 12.2 is backward-compatible)
!pip -q install torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# whisper-timestamped: pin to avoid API drift in refine_whisper_precision / min_word_dur
!pip -q install whisper-timestamped==1.15.4

# MoviePy 2.x changed its import structure; pin 1.x for stability
!pip -q install "moviepy==1.0.3"

!pip -q install pydub pandas

print('\n✓ Installation complete')

In [ ]:
# ── Cell 2: Imports + helpers ─────────────────────────────────────────────────

import gc
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
import whisper_timestamped as wts

# MoviePy 1.x import path (2.x changed this — pinned above to avoid it)
from moviepy.editor import VideoFileClip

from google.colab import files

print('torch        :', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
    print('VRAM (GB)     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

AVAILABLE_MODELS = ['tiny', 'base', 'small', 'medium', 'large']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Single-slot model cache: evict before loading the next model to avoid OOM.
# Colab T4 has ~15 GB VRAM; whisper-large alone needs ~10 GB.
# Keeping two large models loaded simultaneously will crash the session.
_current_model_name: Optional[str] = None
_current_model: Optional[Any] = None


def get_model(name: str) -> Any:
    """Load model, evicting the previous one from VRAM first."""
    global _current_model_name, _current_model

    if name not in AVAILABLE_MODELS:
        raise ValueError(f'Unknown model: {name!r}. Available: {AVAILABLE_MODELS}')

    if _current_model_name == name:
        return _current_model  # already loaded

    # Evict previous model
    if _current_model is not None:
        print(f'  Evicting {_current_model_name!r} from {DEVICE.upper()} memory...')
        del _current_model
        _current_model = None
        _current_model_name = None
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    print(f'  Loading {name!r} on {DEVICE.upper()}...')
    _current_model = wts.load_model(name, device=DEVICE)
    _current_model_name = name
    return _current_model


def extract_audio_if_video(input_path: str) -> str:
    """
    Return a path to an audio file suitable for Whisper.
    If the input is already audio, return it as-is.
    If it's a video, extract the audio to a distinct .wav file.
    """
    p = Path(input_path)
    audio_extensions = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac'}

    if p.suffix.lower() in audio_extensions:
        return str(p)

    # Output to a clearly distinct path to avoid overwriting the source
    out_audio = p.parent / (p.stem + '_extracted.wav')

    clip = VideoFileClip(str(p))
    try:
        if clip.audio is None:
            raise ValueError(f'Uploaded video {p.name!r} has no audio track.')

        duration_min = clip.duration / 60
        if duration_min > 30:
            print(f'  ⚠ Video is {duration_min:.1f} min — transcription may take a long time.')

        # write_audiofile does NOT accept a 'codec' kwarg directly.
        # Pass codec via ffmpeg_params to avoid TypeError.
        clip.audio.write_audiofile(
            str(out_audio),
            fps=44100,
            ffmpeg_params=['-acodec', 'pcm_s16le'],
            logger=None,
        )
    finally:
        clip.close()

    return str(out_audio)


def transcribe_audio(
    audio_path: str,
    model_name: str = 'base',
) -> Tuple[str, List[Dict], float]:
    """Transcribe and return (transcript, segments, elapsed_seconds)."""
    model = get_model(model_name)

    t0 = time.perf_counter()
    result = wts.transcribe(
        model,
        audio_path,
        verbose=False,
        beam_size=5,
        temperature=0.0,
        condition_on_previous_text=False,  # prevents hallucination cascades
        no_speech_threshold=0.6,
        compression_ratio_threshold=2.4,
        language=None,                     # auto-detect
        # whisper-timestamped specific — tighten word-level timestamps
        refine_whisper_precision=0.2,
        min_word_dur=0.02,
    )
    elapsed_s = time.perf_counter() - t0

    transcript: str       = result.get('text', '').strip()
    segments: List[Dict]  = result.get('segments', [])
    return transcript, segments, elapsed_s


def flatten_words(segments: List[Dict]) -> List[Dict]:
    return [w for seg in segments for w in seg.get('words', [])]


print('\n✓ Helpers defined')

In [ ]:
# ── Cell 3: Upload media file ─────────────────────────────────────────────────

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded. Please upload an audio or video file.')

input_file = next(iter(uploaded.keys()))
print(f'\nUploaded  : {input_file}')

audio_file = extract_audio_if_video(input_file)
print(f'Audio path: {audio_file}')

# Show audio duration so the user knows what to expect
import wave, contextlib, subprocess, re
try:
    probe = subprocess.check_output(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', audio_file],
        text=True
    ).strip()
    dur_s = float(probe)
    print(f'Duration  : {dur_s/60:.1f} min ({dur_s:.0f} s)')
except Exception:
    print('(Could not probe duration)')

In [ ]:
# ── Cell 4: Run benchmarks ────────────────────────────────────────────────────
# Models are loaded one at a time; the previous is evicted before the next
# loads to stay within Colab T4 VRAM limits (~15 GB).

MODELS_TO_TEST = ['small', 'medium', 'large']   # edit as needed

for m in MODELS_TO_TEST:
    if m not in AVAILABLE_MODELS:
        raise ValueError(f'Invalid model in MODELS_TO_TEST: {m!r}')

results  = []
artifacts: Dict[str, Any] = {}

for model_name in MODELS_TO_TEST:
    print(f'\n=== {model_name} ===')
    try:
        transcript, segments, elapsed_s = transcribe_audio(audio_file, model_name=model_name)
    except Exception as e:
        print(f'  ✗ Failed: {e}')
        results.append({'model': model_name, 'error': str(e)})
        continue

    words = flatten_words(segments)
    confidences = [w['confidence'] for w in words if w.get('confidence') is not None]
    avg_conf = sum(confidences) / len(confidences) if confidences else None

    # Low-confidence word breakdown
    low_conf = [w for w in words if w.get('confidence', 1.0) < 0.30]

    row = {
        'model'           : model_name,
        'runtime_sec'     : round(elapsed_s, 2),
        'segments'        : len(segments),
        'words'           : len(words),
        'avg_confidence'  : round(avg_conf, 4) if avg_conf is not None else None,
        'low_conf_words'  : len(low_conf),   # words Whisper is uncertain about
        'transcript_chars': len(transcript),
    }
    results.append(row)
    artifacts[model_name] = {'transcript': transcript, 'segments': segments}

    print(f'  runtime   : {elapsed_s:.1f}s')
    print(f'  words     : {len(words)}')
    print(f'  avg conf  : {avg_conf:.3f}' if avg_conf else '  avg conf  : n/a')
    print(f'  low-conf  : {len(low_conf)} words below 0.30')

df = pd.DataFrame([r for r in results if 'error' not in r])
if not df.empty:
    df = df.sort_values('runtime_sec').reset_index(drop=True)
print()
df

In [ ]:
# ── Cell 5: Transcript previews ───────────────────────────────────────────────

for model_name in MODELS_TO_TEST:
    if model_name not in artifacts:
        print(f'\n--- {model_name}: no artifact (errored) ---')
        continue
    text = artifacts[model_name]['transcript']
    preview = (text[:700] + ' [...]') if len(text) > 700 else text
    print(f'\n--- {model_name} ---')
    print(preview)

In [ ]:
# ── Cell 6: Export benchmark JSON ─────────────────────────────────────────────

out_path = 'pleeb_model_benchmark.json'

out = {
    'audio_file': audio_file,
    'device'    : DEVICE,
    'results'   : results,
    'artifacts' : artifacts,
}

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f'Wrote {out_path}')

try:
    files.download(out_path)
except Exception as e:
    print(f'Auto-download failed ({e}) — manually download from the Files panel.')